# DMEPOS by Referring Provider & Services data loading and cleaning

This notebook combines and cleans the Medicare Durable Medical Equipment, Devices & Supplies (DMEPOS) - by Referring Provider and Service datasets from the Centers of Medicare & Medicaid Services (CMS) for program years 2021 through 2023. Data cleaning and preprocessing steps include the imputation of missing values, normalization of text fields, and appending program year to each record.

For information regarding each variable, please refer to the data dictionary downloadable from [DMEPOS - by Referring Provider and Service](https://data.cms.gov/provider-summary-by-type-of-service/medicare-durable-medical-equipment-devices-supplies/medicare-durable-medical-equipment-devices-supplies-by-referring-provider-and-service).

Cells converted to RAW NB Convert are not necessary to run to understand work performed nor run this notebook without exception.

## Import raw files

In [1]:
import pandas as pd

files_names = ['/dsa/groups/casestudycf25/team02/mup_dme_ry25_p05_v10_dy21_rfrhpr.csv',
               '/dsa/groups/casestudycf25/team02/mup_dme_ry25_p05_v10_dy22_rfrhpr.csv',
               '/dsa/groups/casestudycf25/team02/mup_dme_ry25_p05_v10_dy23_rfrhpr.csv']

rfrhpr = {}

dy = 20
for file_name in files_names:
    dy += 1

    # Create DataFrame from the file content
    df = pd.read_csv(file_name,dtype={'Rfrg_Prvdr_State_FIPS':str,'Rfrg_Prvdr_Zip5':str}) # ensure Rfrg_Prvdr_State_FIPS & Rfrg_Prvdr_Zip5 are imported as str
    rfrhpr[dy] = df

Verify the number of records in each file.

In [2]:
len21 = len(rfrhpr[21])
len22 = len(rfrhpr[22])
len23 = len(rfrhpr[23])
print(f'{len21}, {len22}, {len23}')

1516153, 1457378, 1439587


### Change these to code cells and run for additional validation.

## Combine and Clean Datasets

In [3]:
# combine dfs
df_raw = pd.concat([rfrhpr[21],rfrhpr[22],rfrhpr[23]],axis=0)
df_raw.head()

,Rfrg_NPI,Rfrg_Prvdr_Last_Name_Org,Rfrg_Prvdr_First_Name,Rfrg_Prvdr_MI,Rfrg_Prvdr_Crdntls,Rfrg_Prvdr_Ent_Cd,Rfrg_Prvdr_St1,Rfrg_Prvdr_St2,Rfrg_Prvdr_City,Rfrg_Prvdr_State_Abrvtn,...,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplrs,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
0,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,"Portable gaseous oxygen system, rental; includ...",Y,5,NaN,16,16,46.336250,20.097500,14.857500,15.280000
1,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,"Oxygen concentrator, single delivery port, cap...",Y,6,NaN,19,19,360.770000,98.223158,72.843684,79.753158
2,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Standard hemi (low seat) wheelchair,Y,1,NaN,11,11,92.000000,39.230000,31.385455,33.552727
3,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,"Elevating leg rests, pair (for use with capped...",Y,1,NaN,11,11,20.000000,10.210909,8.169091,8.456364
4,1003000480,Rothchild,Kevin,B,MD,I,12605 E 16th Ave,NaN,Aurora,CO,...,"Oxygen concentrator, single delivery port, cap...",Y,4,NaN,11,13,272.003846,80.513846,64.407692,84.701538


In [4]:
# check columns which have missing data
df_nulltots = df_raw.isnull().sum()
df_nulltots
# blank first names can be assumed to be entity type code = ‘O’

Rfrg_NPI                          0
Rfrg_Prvdr_Last_Name_Org          0
Rfrg_Prvdr_First_Name            35
Rfrg_Prvdr_MI               1267759
Rfrg_Prvdr_Crdntls           179868
Rfrg_Prvdr_Ent_Cd                 0
Rfrg_Prvdr_St1                    0
Rfrg_Prvdr_St2              3166448
Rfrg_Prvdr_City                   0
Rfrg_Prvdr_State_Abrvtn           0
Rfrg_Prvdr_State_FIPS             0
Rfrg_Prvdr_Zip5                   0
Rfrg_Prvdr_RUCA_Cat            3509
Rfrg_Prvdr_RUCA                3509
Rfrg_Prvdr_RUCA_Desc           3509
Rfrg_Prvdr_Cntry                  0
Rfrg_Prvdr_Spclty_Cd          36480
Rfrg_Prvdr_Spclty_Desc            0
Rfrg_Prvdr_Spclty_Srce            0
RBCS_Lvl                          0
RBCS_Id                           0
RBCS_Desc                         0
HCPCS_CD                          0
HCPCS_Desc                        0
Suplr_Rentl_Ind                   0
Tot_Suplrs                        0
Tot_Suplr_Benes             3077843
Tot_Suplr_Clms              

### Convert the following cell to code and run to see how we determined imputing null credentials was inappropriate.

### Investigation of null specialty codes
The field `Rfrg_Prvdr_Spclty_Cd` indicating the numeric referring provider specialty code from CMS's Medicare Provider and Supplier Taxonomy Crosswalk has null values; whereas, the field `Rfrg_Prvdr_Spclty_Desc` indicating a text description of the referring provider specialty does not. We investigate the descriptions associated with null specialty codes to determine a rational method of imputation.

In [5]:
# descriptions of missing specialty codes
pd.Series(df_raw[df_raw.Rfrg_Prvdr_Spclty_Cd.isnull()]['Rfrg_Prvdr_Spclty_Desc'].unique()).sort_values()

8                                        Anesthesiology
16                                        Clinic/Center
43                           Clinical Neuropsychologist
10                               Colon & Rectal Surgery
12                                            Counselor
30                                              Dentist
23                                   Emergency Medicine
2                                       Family Medicine
27                          General Acute Care Hospital
5                                      General Practice
24                      Health Maintenance Organization
44                                          Home Health
46                                 Integrative Medicine
0                                     Internal Medicine
6                                        Legal Medicine
22                             Licensed Practical Nurse
42                         Local Education Agency (LEA)
37                                              

#### The following cell was converted to markdown for documentation purposes.
Initial investigation missed the specialty descriptions 'Internal Medicine,' 'General Practice,' 'Anesthesiology', 'Emergency Medicine', 'Otolaryngology', 'Dentist', and 'Obstetrics & Gynecology' which are associated with missing specialty codes.

#### Convert the following cell to code and run to see how we determined that speciatly "Psychologist, Clinical" maps to two codes.

### Perform text normalization on provider credentials

In [6]:
import re
# df_raw['Rfrg_Prvdr_Crdntls'].sort_values()
# replace NaN credentials with empty string
df_raw['Rfrg_Prvdr_Crdntls'] = df_raw['Rfrg_Prvdr_Crdntls'].fillna('')
# normalize credentials to lowercase and remove punctuation
df_raw['Rfrg_Prvdr_Crdntls'] = df_raw['Rfrg_Prvdr_Crdntls'].apply(lambda x: re.sub(r'[^a-zA-Z]','',x).lower())
display(df_raw['Rfrg_Prvdr_Crdntls'].unique())
print(df_raw['Rfrg_Prvdr_Crdntls'].nunique())

array(['md', 'do', 'arnp', ..., 'dpmpsc', 'rnlpcfnp', 'msnanpaprnbc'],
      dtype=object)

3486


### Perform data cleaning on geographic attributes

#### Use the 2010 Rural-Urban Commuting Area Codes and ZIP codes table from the US Department of Agriculture to build a mapping of zip codes to rural-urban commuting area (RUCA) codes for imputing null RUCA values.

In [7]:
# build mapping of zip codes to RUCA
# use 2010 data to keep consistent with methodology per data dictionary
import csv

zip_map = {}

with open('/dsa/groups/casestudycf25/team02/RUCA2010zipcode.csv', mode='r', newline='') as file:
    csv_reader = csv.DictReader(file)
    # print(csv_reader.fieldnames)
    # row = next(csv_reader)
    # print(re.sub(r"'","",row[csv_reader.fieldnames[0]]))
    for row in csv_reader:
        # write RUCA2 code (value) into zip_map with ZIP_CODE as key
        zip_map[re.sub(r"'","",row[csv_reader.fieldnames[0]])] = float(row['RUCA2']) # Access data using column names


In [8]:
# build mapping of RUCA to urban-rural designation and description per https://www.ers.usda.gov/data-products/rural-urban-commuting-area-codes
code_map = {1.:['Urban','Metropolitan area core: primary flow within an urbanized area of 50,000 and greater'],
            1.1:['Urban','Secondary flow 30% to <50% to a larger urbanized area of 50,000 and greater'],
            2.:['Urban','Metropolitan area high commuting: primary flow 30% or more to a urbanized area of 50,000 and greater'],
            2.1:['Urban','Secondary flow 30% to <50% to a larger urbanized area of 50,000 and greater'],
            3.:['Urban','Metropolitan area low commuting: primary flow 10% to <30% to a urbanized area of 50,000 and greater'],
            4.:['Urban','Micropolitan area core: primary flow within an urban cluster of 10,000 to 49,999'],
            4.1:['Urban','Secondary flow 30% to <50% to a urbanized area of 50,000 and greater'],
            5.:['Urban','Micropolitan high commuting: primary flow 30% or more to a urban cluster of 10,000 to 49,999'],
            5.1:['Urban','Secondary flow 30% to <50% to a urbanized area of 50,000 and greater'],
            6.:['Urban','Micropolitan low commuting: primary flow 10% to <30% to a urban cluster of 10,000 to 49,999'],
            7.:['Urban','Small town core: primary flow within an urban cluster of 2,500 to 9,999'],
            7.1:['Urban','Secondary flow 30% to <50% to a urbanized area of 50,000 and greater'],
            7.2:['Urban','Secondary flow 30% to <50% to a urban cluster of 10,000 to 49,999'],
            8.:['Urban','Small town high commuting: primary flow 30% or more to a urban cluster of 2,500 to 9,999'],
            8.1:['Urban','Secondary flow 30% to <50% to a urbanized area of 50,000 and greater'],
            8.2:['Urban','Secondary flow 30% to <50% to a urban cluster of 10,000 to 49,999'],
            9.:['Urban','Small town low commuting: primary flow 10% to <30% to a urban cluster of 2,500 to 9,999'],
            10.:['Rural','Rural areas: primary flow to a tract outside a urbanized area of 50,000 and greater or UC'],
            10.1:['Rural','Secondary flow 30% to <50% to a urbanized area of 50,000 and greater'],
            10.2:['Rural','Secondary flow 30% to <50% to a urban cluster of 10,000 to 49,999'],
            10.3:['Rural','Secondary flow 30% to <50% to a urban cluster of 2,500 to 9,999'],
            99.:['Unknown','Unknown']}
            

In [9]:
missing_zips = df_raw[df_raw.Rfrg_Prvdr_RUCA_Cat.isnull()]['Rfrg_Prvdr_Zip5'].unique()
missing_zips

array(['85222', '21264', '85273', '96278', '20307', '99588', '38163',
       '35870', '83886', '96306', '01016', '46258', '09617', '96271',
       '09574', '21423', '23291', '20462', '11111', '23710', '75073',
       '99574', '85239', '75284', '09094', '09180', '22334', '96273',
       '09464', '80262', '96264', '03681', '10064', '09012', '85218',
       '11535', '50200', '20596', '44193', '85242', '86306', '02192',
       '97313', '97881', '31939', '58303', '75258', '96617', '40852',
       '46092', '96362', '23521', '39126', '17712', '37992', '09112',
       '18154', '31514', '00115', '08228', '02031', '09175', '94410',
       '32886', '10571', '60059', '00000', '54502', '48277', '61837',
       '85228', '09824', '99686', '34595', '27759', '71605', '32323',
       '96328', '09104', '29277', '31610', '99286', '99243', '48351',
       '10087', '98341', '85232', '72836', '96368', '96310', '96350',
       '86027', '96224', '02154', '56703', '63195', '76560', '02078',
       '39432', '008

In [10]:
# some zip codes are missing from RUCA2010zipcode.csv
for missing_zip in missing_zips:
    try:
        print(f"{missing_zip} maps to {zip_map[missing_zip]}")
    except:
        zip_map[missing_zip] = 99. # map zip code to 99 for unknown

21264 maps to 1.0
99588 maps to 10.0
38163 maps to 1.0
23291 maps to 1.0
99574 maps to 10.0
75284 maps to 1.0
22334 maps to 1.0
80262 maps to 1.0
44193 maps to 1.0
32886 maps to 1.0
48277 maps to 1.0
99686 maps to 10.0
32323 maps to 2.0
10087 maps to 1.0
63195 maps to 1.0
02241 maps to 1.0
06030 maps to 1.0
06927 maps to 1.0
06102 maps to 1.0
06050 maps to 1.0
06409 maps to 1.0
06504 maps to 1.0
06520 maps to 1.0
06856 maps to 1.0
06134 maps to 1.0
06250 maps to 4.1
06039 maps to 10.0
06904 maps to 1.0
46277 maps to 1.0
06487 maps to 1.0
06782 maps to 1.0
06777 maps to 10.0
06230 maps to 2.0
06890 maps to 1.0
06824 maps to 1.0
06825 maps to 1.0
35246 maps to 1.0
44194 maps to 1.0
48267 maps to 1.0


In [11]:
# link missing RUCA by zip code from table at https://www.ers.usda.gov/data-products/rural-urban-commuting-area-codes
        
df_clean = df_raw
df_clean.Rfrg_Prvdr_RUCA = df_clean.Rfrg_Prvdr_RUCA.fillna(df_clean.Rfrg_Prvdr_Zip5.map(zip_map))

In [12]:
# map in missing RUCA_Cat and Desc
target_columns = ['Rfrg_Prvdr_RUCA_Cat', 'Rfrg_Prvdr_RUCA_Desc']

for i, col in enumerate(target_columns):
    df_clean[col] = df_clean.apply(lambda row: code_map[row['Rfrg_Prvdr_RUCA']][i] if pd.isna(row[col]) else row[col], axis=1)

### Impute missing specialty codes
The mapping of specialty descriptions to their codes from the Medicare Provider and Supplier Taxonomy Crosswalk published by CMS was created manually based on personal judgement. Any description that lacked an obvious counterpart in the crosswalk or that was so ambiguous as to potentially match several specialty codes was left blank.

In [13]:
# manually map specialty codes to Rfrg_Prvdr_Spclty_Desc using Medicare Provider and Supplier Taxonomy Crosswalk dataset from
# https://data.cms.gov/provider-characteristics/medicare-provider-supplier-enrollment/medicare-provider-and-supplier-taxonomy-crosswalk
specialty_map = {'Anesthesiology': '5','Clinic/Center': '70','Clinical Neuropsychologist': '86','Colon & Rectal Surgery': '28','Counselor': 'E2','Dentist': 'C5','Emergency Medicine': '93','Family Medicine': '8',
                 'General Acute Care Hospital': 'A0','General Practice': '1','Health Maintenance Organization': '58','Home Health': 'A4','Integrative Medicine': '',
                 'Internal Medicine': '11','Legal Medicine': '','Licensed Practical Nurse': '89','Local Education Agency (LEA)': '','Midwife': '42',
                 'Military Health Care Provider': 'A0','Naturopath': '','Neurological Surgery': '14','Neuromusculoskeletal Medicine, Sports Medicine': '23','Obstetrics & Gynecology': '16',
                 'Oral & Maxillofacial Surgery': '19','Orthopaedic Surgery': '20','Otolaryngology': '4','Pain Medicine': '72','Pediatrics': '37','Pedorthist': 'B2',
                 'Personal Emergency Response Attendant': '','Pharmacist': 'A5','Pharmacy Technician': 'A5','Physical Medicine & Rehabilitation': '25',
                 'Plastic Surgery': '24','Prosthetist': '56','Psychiatry & Neurology': '26','Psychoanalyst': '62','Psychologist': '62',
                 'Registered Nurse': '89','Respite Care': '','Sleep Specialist, PhD': 'C0','Social Worker': '80','Specialist': '',
                 'Specialist/Technologist, Other': '75','Student in an Organized Health Care Education/Training Program': '',
                 'Surgery': '2','Thoracic Surgery (Cardiothoracic Vascular Surgery)': '33'}

In [14]:
df_clean.Rfrg_Prvdr_Spclty_Cd = df_clean.Rfrg_Prvdr_Spclty_Cd.fillna(df_clean.Rfrg_Prvdr_Spclty_Desc.map(specialty_map))

### Impute nulls in the Total Supplier Beneficiaries field

In [15]:
# impute missing values in a rational manner
# Real totals can be determined from data described on this webpage:
# https://www.cms.gov/data-research/cms-data/limited-data-set-lds-files/physician/supplier-procedure-summary-psps
# As it costs $400 to obtain we opt for the imputation methodology of Khoshgoftaar & Johnson i.e. imputing 5 for missing Tot_Benes and Tot_Suplr_Benes

df_clean['Tot_Suplr_Benes'] = df_clean['Tot_Suplr_Benes'].fillna(5)

In [16]:
df_clean.isnull().sum() # make sure remaining nulls are sensible

Rfrg_NPI                          0
Rfrg_Prvdr_Last_Name_Org          0
Rfrg_Prvdr_First_Name            35
Rfrg_Prvdr_MI               1267759
Rfrg_Prvdr_Crdntls                0
Rfrg_Prvdr_Ent_Cd                 0
Rfrg_Prvdr_St1                    0
Rfrg_Prvdr_St2              3166448
Rfrg_Prvdr_City                   0
Rfrg_Prvdr_State_Abrvtn           0
Rfrg_Prvdr_State_FIPS             0
Rfrg_Prvdr_Zip5                   0
Rfrg_Prvdr_RUCA_Cat               0
Rfrg_Prvdr_RUCA                   0
Rfrg_Prvdr_RUCA_Desc              0
Rfrg_Prvdr_Cntry                  0
Rfrg_Prvdr_Spclty_Cd              0
Rfrg_Prvdr_Spclty_Desc            0
Rfrg_Prvdr_Spclty_Srce            0
RBCS_Lvl                          0
RBCS_Id                           0
RBCS_Desc                         0
HCPCS_CD                          0
HCPCS_Desc                        0
Suplr_Rentl_Ind                   0
Tot_Suplrs                        0
Tot_Suplr_Benes                   0
Tot_Suplr_Clms              

## Write-out and validate combined datasets

In [17]:
# write to csv
df_clean.to_csv('/dsa/groups/casestudycf25/team02/DMEPOS_rfrhpr_clean.csv', index=False) 

In [20]:
# read clean csv back in for validation
# import pandas as pd

df = pd.read_csv('/dsa/groups/casestudycf25/team02/DMEPOS_rfrhpr_clean.csv',dtype={'Rfrg_Prvdr_State_FIPS':str,'Rfrg_Prvdr_Zip5':str}) # ensure Rfrg_Prvdr_State_FIPS & Rfrg_Prvdr_Zip5 are imported as str
df.head()

,Rfrg_NPI,Rfrg_Prvdr_Last_Name_Org,Rfrg_Prvdr_First_Name,Rfrg_Prvdr_MI,Rfrg_Prvdr_Crdntls,Rfrg_Prvdr_Ent_Cd,Rfrg_Prvdr_St1,Rfrg_Prvdr_St2,Rfrg_Prvdr_City,Rfrg_Prvdr_State_Abrvtn,...,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplrs,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,"Portable gaseous oxygen system, rental; includ...",Y,5,5.0,16,16,46.336250,20.097500,14.857500,15.280000
1,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,"Oxygen concentrator, single delivery port, cap...",Y,6,5.0,19,19,360.770000,98.223158,72.843684,79.753158
2,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Standard hemi (low seat) wheelchair,Y,1,5.0,11,11,92.000000,39.230000,31.385455,33.552727
3,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,"Elevating leg rests, pair (for use with capped...",Y,1,5.0,11,11,20.000000,10.210909,8.169091,8.456364
4,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,NaN,Aurora,CO,...,"Oxygen concentrator, single delivery port, cap...",Y,4,5.0,11,13,272.003846,80.513846,64.407692,84.701538


Verify the number of records was preserved.

In [21]:
check_len = len21 + len22 + len23
df_len = len(df)
print(f'{check_len} versus {df_len}')

4413118 versus 4413118


### We forgot to append years. Let's do that here.

In [22]:
import numpy as np
# arrays of year values
yrs_21 = np.full(len21, 2021)
yrs_22 = np.full(len22, 2022)
yrs_23 = np.full(len23, 2023)

# concatenate
yrs = np.concatenate((yrs_21, yrs_22, yrs_23))

# add year column to the df
df['Year'] = yrs

df.head()


,Rfrg_NPI,Rfrg_Prvdr_Last_Name_Org,Rfrg_Prvdr_First_Name,Rfrg_Prvdr_MI,Rfrg_Prvdr_Crdntls,Rfrg_Prvdr_Ent_Cd,Rfrg_Prvdr_St1,Rfrg_Prvdr_St2,Rfrg_Prvdr_City,Rfrg_Prvdr_State_Abrvtn,...,Suplr_Rentl_Ind,Tot_Suplrs,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt,Year
0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Y,5,5.0,16,16,46.336250,20.097500,14.857500,15.280000,2021
1,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Y,6,5.0,19,19,360.770000,98.223158,72.843684,79.753158,2021
2,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Y,1,5.0,11,11,92.000000,39.230000,31.385455,33.552727,2021
3,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Y,1,5.0,11,11,20.000000,10.210909,8.169091,8.456364,2021
4,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,NaN,Aurora,CO,...,Y,4,5.0,11,13,272.003846,80.513846,64.407692,84.701538,2021


In [23]:
# overwrite the csv
df.to_csv('/dsa/groups/casestudycf25/team02/DMEPOS_rfrhpr_clean.csv', index=False) 

### Convert the following cell to code and run to see the data types of the full dataset.